# Reranker Exploration — From First Principles

The goal here is to build intuition about what a reranker actually does and how its scores behave — before relying on them for adaptive top-k cutoffs.

Questions we want to answer:
- What score does a reranker assign to a clearly relevant document vs. a clearly irrelevant one?
- Is the score range consistent, or does it shift by query?
- Where does the "natural gap" appear in a ranked list — and is it reliable?
- Does the gap separate gold citations from non-gold ones cleanly?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

plt.style.use('fivethirtyeight')
plt.rcParams['figure.dpi'] = 110

try:
    BASE = '/kaggle/input/llm-agentic-legal-information-retrieval'
    open(f'{BASE}/train.csv')
except FileNotFoundError:
    BASE = '../data'

RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
model.eval().to(DEVICE)
print('Model loaded.')

In [ ]:
@torch.no_grad()
def rerank_score(query: str, documents: list[str]) -> np.ndarray:
    """Return a relevance score for each (query, doc) pair."""
    pairs = [[query, doc] for doc in documents]
    inputs = tokenizer(
        pairs,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    ).to(DEVICE)
    logits = model(**inputs).logits.squeeze(-1)
    return torch.sigmoid(logits).cpu().numpy()

## Example 1 — Controlled toy examples

Before touching real data, let's see how scores behave on synthetic examples where we **know** the answer.

Query: a clear legal question. Documents range from an exact match to total noise.

In [ ]:
query_toy = "What are the conditions for ordering pre-trial detention under Swiss law?"

docs_toy = [
    # Clearly relevant — directly answers the question
    "Untersuchungshaft darf nur angeordnet werden, wenn die beschuldigte Person eines Verbrechens oder Vergehens dringend verdächtig ist und ernsthafte Bedenken bestehen, dass sie sich durch Flucht der Strafverfolgung entzieht, oder die Gefahr besteht, dass sie Personen beeinflusst oder Beweise vernichtet.",
    # Relevant but partial — mentions detention but not all conditions
    "Die Haftdauer darf die mutmassliche Dauer der zu erwartenden Freiheitsstrafe nicht übersteigen.",
    # Topically related — criminal procedure, but different subject
    "Die beschuldigte Person hat das Recht, einen Verteidiger beizuziehen.",
    # Same domain, completely different topic
    "Das Gericht kann eine Busse bis zu 1000 Franken verhängen.",
    # Total noise — unrelated law
    "Der Eigentümer einer Sache kann diese von jedem, der sie ihm vorenthält, herausverlangen.",
    # Gibberish
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit.",
]

labels_toy = [
    'Exact match (detention conditions)',
    'Partial (detention duration)',
    'Related (right to counsel)',
    'Same domain, different topic (fine)',
    'Different domain (property law)',
    'Gibberish',
]

scores_toy = rerank_score(query_toy, docs_toy)

for label, score in zip(labels_toy, scores_toy):
    print(f'{score:.4f}  {label}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green', 'yellowgreen', 'gold', 'orange', 'tomato', 'gray']
bars = ax.barh(labels_toy[::-1], scores_toy[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='0.5 threshold')
for bar, score in zip(bars, scores_toy[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=10)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Reranker score (sigmoid of logit)')
ax.set_title('Toy example — score by relevance tier')
ax.legend()
plt.tight_layout()
plt.show()

## Example 1b — Same docs, German query\n\nThe scores above are suspiciously low. Cross-encoder rerankers see query+document concatenated — if they're in different languages, attention patterns may not align well even for a multilingual model.\n\nRepeat with the German equivalent of the same query to isolate whether language mismatch is the cause."

In [ ]:

# German equivalent of the toy query
query_toy_de = "Was sind die Voraussetzungen für die Anordnung von Untersuchungshaft nach Schweizer Recht?"

# Score all three conditions: EN-query/DE-doc, DE-query/DE-doc, EN-query/EN-doc (control)
docs_en_control = [
    "Pre-trial detention may only be ordered if the accused is strongly suspected of a crime or misdemeanor and there are serious concerns that they will flee prosecution, or there is a risk that they will influence persons or destroy evidence.",
    "The duration of detention may not exceed the expected length of the anticipated custodial sentence.",
    "The accused has the right to engage a defense counsel.",
    "The court may impose a fine of up to 1,000 francs.",
    "The owner of a thing may reclaim it from anyone who withholds it.",
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit.",
]

scores_en_de  = rerank_score(query_toy,    docs_toy)           # EN query / DE docs  ← actual pipeline
scores_de_de  = rerank_score(query_toy_de, docs_toy)           # DE query / DE docs  ← oracle
scores_en_en  = rerank_score(query_toy,    docs_en_control)    # EN query / EN docs  ← monolingual control

comparison = pd.DataFrame({
    'document':  labels_toy,
    'EN-query/DE-doc': scores_en_de,
    'DE-query/DE-doc': scores_de_de,
    'EN-query/EN-doc': scores_en_en,
})
comparison['language_penalty'] = comparison['DE-query/DE-doc'] - comparison['EN-query/DE-doc']
print(comparison.to_string(index=False))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

x = np.arange(len(labels_toy))
w = 0.28

# Left: grouped bar — all three conditions side by side
axes[0].barh(x + w,   comparison['EN-query/EN-doc'], w, label='EN-query/EN-doc (control)', color='steelblue')
axes[0].barh(x,       comparison['DE-query/DE-doc'], w, label='DE-query/DE-doc (oracle)',  color='green')
axes[0].barh(x - w,   comparison['EN-query/DE-doc'], w, label='EN-query/DE-doc (pipeline)',color='tomato')
axes[0].set_yticks(x)
axes[0].set_yticklabels(labels_toy, fontsize=9)
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('Reranker score (sigmoid)')
axes[0].set_title('Score by language pair — all relevance tiers')
axes[0].legend(fontsize=9)

# Right: language penalty (DE-DE minus EN-DE) per document — shows how much is lost
penalty = comparison['language_penalty']
colors_p = ['green' if p >= 0 else 'tomato' for p in penalty]
axes[1].barh(labels_toy[::-1], penalty[::-1], color=colors_p[::-1], edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Score increase when query is in German\n(DE-query/DE-doc  −  EN-query/DE-doc)')
axes[1].set_title('Language penalty on cross-encoder score')

plt.suptitle('Cross-lingual mismatch diagnostic — bge-reranker-v2-m3', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nMean language penalty across all docs:     {penalty.mean():+.4f}")
print(f"Penalty on most-relevant doc only:         {penalty.iloc[0]:+.4f}")
print(f"Penalty on least-relevant doc (gibberish): {penalty.iloc[-1]:+.4f}")


## Example 2 — Real val query, hand-picked docs

Now use an actual val query and manually select:
- A known gold citation (should score high)
- A plausible-looking but wrong article (same law code, wrong article)
- A completely off-topic article

This tells us if the model can distinguish within the same law code — which is the hard case.

In [ ]:
val_df   = pd.read_csv(f'{BASE}/val.csv')
laws_df  = pd.read_csv(f'{BASE}/laws_de.csv')
laws_df['text'] = laws_df['text'].fillna('')

# Pick val query 3 (10 citations — manageable) and its first gold citation
row = val_df.iloc[3]
query_real = row['query']
gold_cits  = [c.strip() for c in row['gold_citations'].split(';')]

print(f'Query (first 400 chars):\n{query_real[:400]}')
print(f'\nGold citations ({len(gold_cits)}): {gold_cits}')

In [ ]:
# Fetch gold articles and a sample of non-gold articles from same law code(s)
gold_texts = laws_df[laws_df['citation'].isin(gold_cits)][['citation','text']].copy()

# Get the law codes present in gold citations
import re
gold_codes = set()
for c in gold_cits:
    m = re.search(r'([A-ZÜÄÖ]{2,})\s*$', c.strip())
    if m:
        gold_codes.add(m.group(1))

# Non-gold articles from the same law codes (hard negatives)
same_code_mask = laws_df['citation'].apply(
    lambda c: any(code in str(c) for code in gold_codes)
) & ~laws_df['citation'].isin(gold_cits)

hard_negatives = laws_df[same_code_mask].sample(5, random_state=42)[['citation','text']]

# Off-topic articles (different law code entirely)
off_topic_mask = laws_df['citation'].apply(
    lambda c: not any(code in str(c) for code in gold_codes)
) & (laws_df['text'].str.len() > 50)
off_topic = laws_df[off_topic_mask].sample(5, random_state=42)[['citation','text']]

# Combine and score
all_docs = pd.concat([
    gold_texts.assign(group='gold'),
    hard_negatives.assign(group='hard_negative'),
    off_topic.assign(group='off_topic'),
], ignore_index=True)

all_docs['score'] = rerank_score(query_real[:2000], all_docs['text'].tolist())
all_docs = all_docs.sort_values('score', ascending=False).reset_index(drop=True)

print(f'Gold law codes: {gold_codes}\n')
print(all_docs[['group','citation','score']].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

color_map = {'gold': 'green', 'hard_negative': 'orange', 'off_topic': 'tomato'}
colors = [color_map[g] for g in all_docs['group']]
labels_plot = [f"{row['citation']} [{row['group']}]" for _, row in all_docs.iterrows()]

ax.barh(labels_plot[::-1], all_docs['score'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2)
ax.set_xlabel('Reranker score')
ax.set_title('Real val query — gold vs hard negatives vs off-topic')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='green',  label='gold citation'),
    Patch(color='orange', label='hard negative (same law code)'),
    Patch(color='tomato', label='off-topic'),
])
plt.tight_layout()
plt.show()

## Example 3 — Score distribution across all laws for one query

Now score the query against **all** laws in the corpus and plot the full distribution. 

This shows:
- Where gold citations land in the overall ranking
- Whether there's a natural score gap between gold and non-gold
- Whether a simple threshold (e.g. 0.5) or a relative gap is more reliable

In [ ]:
# Score all laws — may take a few minutes on CPU
# To keep runtime manageable, score top-200 candidates from a BM25/embedding pre-filter
# Here we randomly sample 300 non-gold + all gold to approximate the reranking step

non_gold = laws_df[~laws_df['citation'].isin(gold_cits) & (laws_df['text'].str.len() > 20)]
sample_non_gold = non_gold.sample(300, random_state=42)[['citation','text']]

eval_pool = pd.concat([
    gold_texts[['citation','text']].assign(is_gold=True),
    sample_non_gold.assign(is_gold=False),
], ignore_index=True)

print(f'Scoring {len(eval_pool)} documents...')
BATCH = 32
scores_all = []
for i in range(0, len(eval_pool), BATCH):
    batch_docs = eval_pool['text'].iloc[i:i+BATCH].tolist()
    scores_all.extend(rerank_score(query_real[:2000], batch_docs).tolist())

eval_pool['score'] = scores_all
eval_pool = eval_pool.sort_values('score', ascending=False).reset_index(drop=True)
eval_pool['rank'] = eval_pool.index + 1

print('\nTop-20:')
print(eval_pool[['rank','citation','is_gold','score']].head(20).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: score by rank, highlight gold
non_gold_pool = eval_pool[~eval_pool['is_gold']]
gold_pool     = eval_pool[eval_pool['is_gold']]

axes[0].scatter(non_gold_pool['rank'], non_gold_pool['score'],
                s=15, alpha=0.4, color='steelblue', label='non-gold')
axes[0].scatter(gold_pool['rank'], gold_pool['score'],
                s=80, color='green', zorder=5, label='gold citation')
axes[0].axhline(0.5, color='black', linestyle='--', linewidth=1, label='0.5')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Reranker score')
axes[0].set_title('Score by rank — gold vs non-gold')
axes[0].legend()

# Right: score gap between consecutive ranks (top 30)
top30 = eval_pool.head(30).copy()
top30['gap'] = top30['score'].diff(-1).fillna(0)  # drop from this rank to next

bar_colors = ['green' if g else 'steelblue' for g in top30['is_gold']]
axes[1].bar(top30['rank'], top30['gap'], color=bar_colors, edgecolor='white')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Score drop to next rank')
axes[1].set_title('Score gap between consecutive ranks (top 30)')

from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color='green',     label='gold citation at this rank'),
    Patch(color='steelblue', label='non-gold'),
])

plt.tight_layout()
plt.show()

# Where do gold citations rank?
print('Gold citation ranks and scores:')
print(gold_pool[['rank','citation','score']].to_string(index=False))

## Example 4 — Score gap across all val queries

Repeat the above for all 10 val queries (against a sampled pool) and ask: 
- Is there a consistent gap after the last gold citation?
- Does the gap size correlate with citation count?
- At what score threshold do gold citations stop appearing?

In [ ]:
results = []

for idx, row in val_df.iterrows():
    query    = row['query'][:2000]
    gold_set = set(c.strip() for c in row['gold_citations'].split(';'))
    gold_arts = laws_df[laws_df['citation'].isin(gold_set)][['citation','text']]
    non_gold_sample = laws_df[
        ~laws_df['citation'].isin(gold_set) & (laws_df['text'].str.len() > 20)
    ].sample(200, random_state=idx)[['citation','text']]

    pool = pd.concat([
        gold_arts.assign(is_gold=True),
        non_gold_sample.assign(is_gold=False),
    ], ignore_index=True)

    scores = []
    for i in range(0, len(pool), BATCH):
        scores.extend(rerank_score(query, pool['text'].iloc[i:i+BATCH].tolist()).tolist())
    pool['score'] = scores
    pool = pool.sort_values('score', ascending=False).reset_index(drop=True)
    pool['rank'] = pool.index + 1

    last_gold_rank  = pool[pool['is_gold']]['rank'].max()
    last_gold_score = pool[pool['is_gold']]['score'].min()
    first_non_after = pool[(~pool['is_gold']) & (pool['rank'] > last_gold_rank)]['score']
    gap_after_last_gold = last_gold_score - first_non_after.max() if len(first_non_after) else np.nan
    min_gold_score = pool[pool['is_gold']]['score'].min()
    
    results.append({
        'val_idx': idx,
        'n_gold': len(gold_set),
        'last_gold_rank': last_gold_rank,
        'last_gold_score': last_gold_score,
        'gap_after_last_gold': gap_after_last_gold,
        'min_gold_score': min_gold_score,
        'any_gold_below_0_5': (pool[pool['is_gold']]['score'] < 0.5).any(),
        'pool': pool,
    })
    print(f'Query {idx}: {len(gold_set)} gold | last gold @ rank {last_gold_rank} | '
          f'min gold score {min_gold_score:.3f} | gap {gap_after_last_gold:.3f}')

summary = pd.DataFrame([{k:v for k,v in r.items() if k != 'pool'} for r in results])
print('\nSummary:')
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 10))

for ax, res in zip(axes.flat, results):
    pool = res['pool'].head(60)
    non_g = pool[~pool['is_gold']]
    gold_p = pool[pool['is_gold']]

    ax.scatter(non_g['rank'], non_g['score'], s=12, alpha=0.4, color='steelblue')
    ax.scatter(gold_p['rank'], gold_p['score'], s=60, color='green', zorder=5)
    ax.axhline(0.5, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f"Query {res['val_idx']} ({res['n_gold']} gold)", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Rank', fontsize=8)
    ax.set_ylabel('Score', fontsize=8)

plt.suptitle('Reranker scores — gold (green) vs non-gold (blue), all val queries', fontsize=13)
plt.tight_layout()
plt.show()